# BERTopic over the FoodScholar chunk corpus — an exploratory bake-off

**Goal.** Run [BERTopic](https://maartengr.github.io/BERTopic/) over the chunk corpus and see
what an off-the-shelf, fully unsupervised topic model finds — then contrast it with
**our method** (FoodScholar's Layer A + Layer B), qualitatively *and* quantitatively.

**Our method, in one paragraph.** FoodScholar doesn't do flat topic modeling. It first builds
**Layer A** — a *browsable backbone* projected from the **FoodOn** food ontology (an is-a tree
of shelves like *Seafood → Bivalve mollusks*). Then **Layer B** discovers *themes within each
shelf* via a dual pass: (1) mutual-kNN on **BGE-base** chunk embeddings + **Leiden** community
detection, merged with (2) a graph of **shared FoodOn entities** (entity co-occurrence) + Leiden,
labeled by c-TF-IDF. So topics are *grounded in an ontology*, *hierarchical*, *per-shelf*, and
*entity-aware*. BERTopic, by contrast, is global, flat-then-hierarchical, and driven only by text.

**This notebook is standalone exploration / research** — it does **not** rebuild or modify the
library pipeline, and needs **no LLM / API key**. To keep the comparison fair we feed BERTopic the
**same BGE-base vectors** Layer B uses, so any difference is method, not embeddings.

> Run in the **`foodscholar`** conda env (Python 3.11). First run installs `bertopic`,
> `umap-learn`, `plotly` and downloads the BGE-base model (~400 MB); embeddings are cached so
> reruns are instant.

## 1 · Setup & dependencies

In [1]:
# bertopic / umap-learn / plotly aren't in the foodscholar env by default.
# hdbscan, scikit-learn, sentence-transformers and torch already are.
try:
    import bertopic  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "bertopic", "umap-learn", "plotly"], check=True)

/mnt/miniconda3/envs/foodscholar/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json
from collections import Counter
from itertools import combinations
import math

import numpy as np
import pandas as pd
from pathlib import Path

# Same ROOT convention as graph_build.ipynb (works whether cwd is repo root or notebooks/).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE = ROOT / "data" / "cache"; CACHE.mkdir(parents=True, exist_ok=True)
OUT = ROOT / "notebooks" / "out"; OUT.mkdir(parents=True, exist_ok=True)
SNAPSHOT = ROOT / "data" / "annotated.parquet"
FOODON_CACHE = ROOT / "data" / "foodon_cache.parquet"

RANDOM_STATE = 42
# Set to an int (e.g. 800) for a fast iteration pass; None uses the full corpus.
SAMPLE = None
print("ROOT:", ROOT)

ROOT: /mnt/workspaces/wisefood/foodscholar-lib


## 2 · Load the corpus

In [3]:
df = pd.read_parquet(SNAPSHOT)
df = df[df["text"].str.len() >= 50].reset_index(drop=True)   # drop near-empty chunks
if SAMPLE:
    df = df.head(SAMPLE).reset_index(drop=True)

docs = df["text"].tolist()
chunk_ids = df["chunk_id"].tolist()

has_fo = df["foodon_ids"].apply(lambda x: x is not None and len(x) > 0)
print(f"{len(docs):,} chunks")
print("source_type:", df["source_type"].value_counts().to_dict())
print(f"chunks with FoodOn entity links: {int(has_fo.sum()):,} ({has_fo.mean():.0%})")

13,252 chunks
source_type: {'textbook': 12121, 'guide': 1131}
chunks with FoodOn entity links: 6,276 (47%)


## 3 · Embeddings — BGE-base, cached

We embed with `BAAI/bge-base-en-v1.5`, L2-normalized — **identical** to FoodScholar's
`HFEmbedder` (the production chunk embedder Layer B Pass 1 runs on). Vectors are cached to
`data/cache/` keyed by the exact chunk-id list, so reruns load instantly and stay aligned.

In [4]:
MODEL_NAME = "BAAI/bge-base-en-v1.5"
EMB_PATH = CACHE / f"bge_base_emb_{len(docs)}.npy"
IDS_PATH = CACHE / f"bge_base_ids_{len(docs)}.json"

cache_valid = (EMB_PATH.exists() and IDS_PATH.exists()
               and json.loads(IDS_PATH.read_text()) == chunk_ids)

if cache_valid:
    embeddings = np.load(EMB_PATH)
    print("loaded cached embeddings:", embeddings.shape)
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME)          # == foodscholar HFEmbedder model
    embeddings = model.encode(docs, normalize_embeddings=True, batch_size=64,
                              show_progress_bar=True, convert_to_numpy=True)
    np.save(EMB_PATH, embeddings)
    IDS_PATH.write_text(json.dumps(chunk_ids))
    print("encoded + cached:", embeddings.shape)

loaded cached embeddings: (13252, 768)


## 4 · Fit BERTopic

Explicit sub-models so nothing is hidden, fed our **precomputed embeddings** (so BERTopic skips
its own embedder). UMAP is seeded for determinism. HDBSCAN finds a *natural* number of topics and
sends low-density chunks to an **outlier** bucket (`-1`) — note this contrasts with our method's
*coverage-by-construction* goal.

In [5]:
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric="cosine", random_state=RANDOM_STATE)
hdbscan_model = HDBSCAN(min_cluster_size=30, metric="euclidean",
                        cluster_selection_method="eom", prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=5)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=False,
    verbose=True,
)
topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)
topics = np.asarray(topics)

n_topics = len(set(topics) - {-1})
outlier_frac = float(np.mean(topics == -1))
print(f"\ntopics found: {n_topics}   |   outliers (-1): {outlier_frac:.1%}")

2026-06-04 14:13:52,024 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-04 14:14:39,466 - BERTopic - Dimensionality - Completed ✓
2026-06-04 14:14:39,467 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-04 14:14:40,115 - BERTopic - Cluster - Completed ✓
2026-06-04 14:14:40,122 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-04 14:14:43,072 - BERTopic - Representation - Completed ✓



topics found: 82   |   outliers (-1): 30.0%


## 5 · Qualitative — what did it find?

In [6]:
info = topic_model.get_topic_info()
info[["Topic", "Count", "Name", "Representation"]].head(25)

,Topic,Count,Name,Representation
0,-1,3971,-1_glucose_fruit_vegetables_glycogen,"[glucose, fruit, vegetables, glycogen, sugar, ..."
1,0,658,0_iron_zinc_copper_chromium,"[iron, zinc, copper, chromium, iodine, seleniu..."
2,1,473,1_sustainable_environmental_land_policy,"[sustainable, environmental, land, policy, 201..."
3,2,444,2_cup_oz_cooked_sauce,"[cup, oz, cooked, sauce, cup cooked, medium, b..."
4,3,361,3_folate_biotin_niacin_vitamin 12,"[folate, biotin, niacin, vitamin 12, thiamin, ..."
5,4,335,4_foodborne_raw_clean_illness,"[foodborne, raw, clean, illness, foodborne ill..."
6,5,261,5_breast milk_baby_breast_breastfeeding,"[breast milk, baby, breast, breastfeeding, for..."
7,6,259,6_child_kids_children_snacks,"[child, kids, children, snacks, offer, childs,..."
8,7,242,7_study_research_information_studies,"[study, research, information, studies, scient..."
9,8,224,8_supplements_supplement_herbal_dietary supple...,"[supplements, supplement, herbal, dietary supp..."


In [7]:
# A few topics with representative documents.
for t in info.Topic.head(7):
    if t == -1:
        continue
    words = ", ".join(w for w, _ in topic_model.get_topic(t)[:8])
    print(f"\n=== Topic {t}  ({int(info.loc[info.Topic==t,'Count'].iloc[0])} chunks) ===")
    print("words:", words)
    for d in topic_model.get_representative_docs(t)[:2]:
        print("   •", " ".join(d[:240].split()), "…")


=== Topic 0  (658 chunks) ===
words: iron, zinc, copper, chromium, iodine, selenium, manganese, iron absorption
   • 5. dietary ligands and zinc absorption. J Nutr 1985; 115:411-14. 24. Solomons N, Jacob R. Studies on the bioavailability of zinc in humans: Eff ects of heme and nonheme iron on the absorption of zinc. Am J Clin Nutr 1981; 34:475-82. 25. Zi …
   • 12. V ought R, London W , Lutwak L, Dublin T. Reliability of estimates of serum inorganic iodine and daily fecal and urinary iodine excretion from single casual specimens. J Clin Endocr Metab 1963; - 23:1218-28. 13. Nath SK, Moin …

=== Topic 1  (473 chunks) ===
words: sustainable, environmental, land, policy, 2013, sustainability, impacts, impact
   • - (4) Although 'we absolutely need this public voice,' Eakman said, other nongovernmental actors can help to engender change. Eakman referred to Lang's suggestion that the United Nations could play a role by issuing a …
   • A second key lesson learned from Europe is the need for

In [8]:
# Hierarchical topics — BERTopic's analog to our Layer A backbone (a tree over topics).
hier = topic_model.hierarchical_topics(docs)
print(topic_model.get_topic_tree(hier))

100%|██████████| 81/81 [00:00<00:00, 175.90it/s]


.
├─iron_zinc_deficiency_nutr_copper
│    ├─amino_amino acids_bile_membrane_ldl
│    │    ├─atp_phosphorylation_reaction_coa_nadh
│    │    │    ├─■──allosteric_reaction_glycogen_amp_enzyme ── Topic: 56
│    │    │    └─coa_atp_acetyl_nadh_phosphorylation
│    │    │         ├─■──nadh_phosphorylation_atp_electron_electron transport ── Topic: 51
│    │    │         └─■──coa_acetyl_acetyl coa_acyl_oxaloacetate ── Topic: 68
│    │    └─amino_amino acids_bile_ldl_intestine
│    │         ├─amino_amino acids_glutamine_proteins_radical
│    │         │    ├─■──radical_radicals_reactive_peroxide_hydrogen peroxide ── Topic: 50
│    │         │    └─amino_amino acids_glutamine_proteins_amino acid
│    │         │         ├─chloride_fluid_cell_ph_membrane
│    │         │         │    ├─■──cell_membrane_plasma membrane_organelles_apparatus ── Topic: 44
│    │         │         │    └─■──chloride_fluid_ph_ions_na ── Topic: 12
│    │         │         └─amino_amino acids_glutamine_amino acid_ammon

In [9]:
# Interactive Plotly views, saved to notebooks/out/ and shown inline.
fig_h = topic_model.visualize_hierarchy(hierarchical_topics=hier)
fig_h.write_html(str(OUT / "bertopic_hierarchy.html"))
topic_model.visualize_topics().write_html(str(OUT / "bertopic_intertopic.html"))
topic_model.visualize_barchart(top_n_topics=12).write_html(str(OUT / "bertopic_barchart.html"))
print("saved 3 HTML views to", OUT)
fig_h

saved 3 HTML views to /mnt/workspaces/wisefood/foodscholar-lib/notebooks/out


## 6 · Quantitative

Two families of numbers:

1. **Intrinsic** — topic count, outlier share, and **NPMI topic coherence** (computed dependency-free
   from corpus word co-occurrence over each topic's top-10 words; higher = more coherent, range ≈ [-1, 1]).
2. **FoodOn alignment** — the real *"vs our method"* signal. Our method grounds topics in FoodOn
   entities; here we ask **how much unsupervised BERTopic recovers that grounding on its own**:
   coverage of entity-bearing chunks, NMI between topic assignment and each chunk's dominant FoodOn id,
   and how concentrated each topic is on a few FoodOn entities.

In [10]:
# --- NPMI topic coherence (dependency-free) ---
TOPN = 10
topic_ids = [t for t in info.Topic if t != -1]
top_words = {t: [w for w, _ in topic_model.get_topic(t)[:TOPN]] for t in topic_ids}
vocab = sorted({w for ws in top_words.values() for w in ws})

bin_vec = CountVectorizer(vocabulary=vocab, binary=True, ngram_range=(1, 2))
X = bin_vec.fit_transform(docs).tocsc()        # docs x vocab, binary occurrence
D = X.shape[0]
col = {w: i for i, w in enumerate(vocab)}
df_count = np.asarray(X.sum(axis=0)).ravel()   # doc frequency per word

def _npmi(a, b):
    da, db = df_count[col[a]], df_count[col[b]]
    if da == 0 or db == 0:
        return 0.0
    co = X[:, col[a]].multiply(X[:, col[b]]).nnz
    if co == 0:
        return -1.0
    p_a, p_b, p_ab = da / D, db / D, co / D
    return math.log(p_ab / (p_a * p_b)) / (-math.log(p_ab))

topic_coh = {}
for t in topic_ids:
    pairs = list(combinations(top_words[t], 2))
    topic_coh[t] = float(np.mean([_npmi(a, b) for a, b in pairs])) if pairs else np.nan
mean_npmi = float(np.nanmean(list(topic_coh.values())))
print(f"mean NPMI coherence (top-{TOPN} words): {mean_npmi:.3f}")

mean NPMI coherence (top-10 words): 0.308


In [11]:
# --- FoodOn alignment ---
from sklearn.metrics import normalized_mutual_info_score

fo_label = pd.read_parquet(FOODON_CACHE).set_index("id")["label"].to_dict()
lbl = lambda i: fo_label.get(i, i)

work = df.copy()
work["topic"] = topics
work["fo_dom"] = work["foodon_ids"].apply(lambda x: x[0] if (x is not None and len(x) > 0) else None)

mask = work["fo_dom"].notna()
coverage = float((work.loc[mask, "topic"] != -1).mean())
nmi = float(normalized_mutual_info_score(work.loc[mask, "fo_dom"], work.loc[mask, "topic"]))

def concentration(sub, k=5):
    cnt = Counter(i for ids in sub["foodon_ids"] if ids is not None for i in ids)
    tot = sum(cnt.values())
    return sum(n for _, n in cnt.most_common(k)) / tot if tot else np.nan

conc = float(np.nanmean([concentration(work[work["topic"] == t]) for t in topic_ids]))

print(f"FoodOn-bearing chunks: {int(mask.sum()):,}")
print(f"  · covered by a (non-outlier) topic : {coverage:.1%}")
print(f"  · NMI(topic ; dominant FoodOn id)  : {nmi:.3f}")
print(f"  · mean top-5 FoodOn concentration  : {conc:.1%}  (how 'about-one-thing' a topic is)")

FoodOn-bearing chunks: 6,276
  · covered by a (non-outlier) topic : 68.7%
  · NMI(topic ; dominant FoodOn id)  : 0.359
  · mean top-5 FoodOn concentration  : 39.2%  (how 'about-one-thing' a topic is)


In [12]:
# Bridge table: each topic's text words next to its dominant FoodOn entities (human labels).
rows = []
for t in topic_ids[:20]:
    sub = work[work["topic"] == t]
    cnt = Counter(i for ids in sub["foodon_ids"] if ids is not None for i in ids)
    rows.append({
        "topic": t,
        "size": len(sub),
        "npmi": round(topic_coh[t], 3),
        "top_words": ", ".join(top_words[t][:6]),
        "top_foodon": "; ".join(f"{lbl(i)} ({n})" for i, n in cnt.most_common(5)),
    })
pd.DataFrame(rows)

,topic,size,npmi,top_words,top_foodon
0,0,658,0.337,"iron, zinc, copper, chromium, iodine, selenium",meat (raw) (24); poultry (18); fish species (1...
1,1,473,0.424,"sustainable, environmental, land, policy, 2013...",meat (raw) (49); edible food (40); fruit produ...
2,2,444,0.339,"cup, oz, cooked, sauce, cup cooked, medium",pasta (67); vegetable (59); food calorie datum...
3,3,361,0.372,"folate, biotin, niacin, vitamin 12, thiamin, 12",thiamin mononitrate (20); egg or egg component...
4,4,335,0.377,"foodborne, raw, clean, illness, foodborne illn...",edible food (72); poultry (62); meat (raw) (59...
5,5,261,0.539,"breast milk, baby, breast, breastfeeding, form...",mammalian milk for infant (67); infant formula...
6,6,259,0.234,"child, kids, children, snacks, offer, childs",vegetable (42); cow milk (39); fruit produce (...
7,7,242,0.251,"study, research, information, studies, scienti...",dietary supplement (7); seed gum (4); dietary ...
8,8,224,0.325,"supplements, supplement, herbal, dietary suppl...",dietary supplement (72); edible food (23); die...
9,9,195,0.351,"sweat, dehydration, fluid, sports drinks, wate...",coffee beverage (11); beverage containing caff...


In [13]:
# One consolidated metrics summary.
summary = pd.DataFrame([
    ("chunks",                       len(docs)),
    ("topics (excl. outliers)",      n_topics),
    ("outlier share",               f"{outlier_frac:.1%}"),
    ("median topic size",            int(np.median([(topics == t).sum() for t in topic_ids]))),
    ("mean NPMI coherence",         f"{mean_npmi:.3f}"),
    ("FoodOn coverage (entity chunks)", f"{coverage:.1%}"),
    ("NMI vs dominant FoodOn id",   f"{nmi:.3f}"),
    ("mean top-5 FoodOn concentration", f"{conc:.1%}"),
], columns=["metric", "value"])
summary

,metric,value
0,chunks,13252
1,topics (excl. outliers),82
2,outlier share,30.0%
3,median topic size,79
4,mean NPMI coherence,0.308
5,FoodOn coverage (entity chunks),68.7%
6,NMI vs dominant FoodOn id,0.359
7,mean top-5 FoodOn concentration,39.2%


## 7 · How this compares to our method

Read the table against the numbers above.

| Dimension | **BERTopic (this notebook)** | **FoodScholar — Layer A + B** |
|---|---|---|
| **Structure** | Flat topics, plus an optional hierarchy *derived* from c-TF-IDF distances | Ontology **is-a backbone** (Layer A) with per-shelf themes (Layer B) — hierarchy is *given*, not inferred |
| **Grounding** | Data-driven words only | Grounded in **FoodOn entity ids** — every theme traces to ontology terms + source chunks |
| **Entity signal** | Ignored | Layer B **Pass 2** clusters on *shared FoodOn entities* — a signal BERTopic has no access to |
| **Coverage** | Drops low-density chunks to an **outlier** bucket (see *outlier share*) | **Coverage by construction** — every mentioned food is kept; every shelf gets themes |
| **Granularity** | Global topics over the whole corpus | **Per-shelf** themes (3–12 per shelf) — finer, browsing-oriented |
| **Determinism** | Seeded UMAP (some variance remains) | Fixed Leiden seed (42) — identical inputs → identical themes |
| **Naming** | c-TF-IDF keywords | c-TF-IDF + optional LLM polish *and* a human FoodOn alias (e.g. `FOODON:00002263` → *Bivalve mollusks*) |

**Interpretation prompts** (fill in from your run):
- Does BERTopic's hierarchy (§5 tree) echo FoodOn's families, or cut the corpus a different way?
- The **NMI / concentration** numbers say how much BERTopic *re-discovers* FoodOn grouping unsupervised —
  low values are the quantitative case for our ontology-grounded approach; high values suggest the text
  alone already carries much of the structure.
- The **outlier share** is the cost BERTopic pays that Layer A/B avoids by construction.

## 8 · Optional — topic-count sensitivity

BERTopic's docs stress that the **number of topics** strongly shapes quality. This sweeps
`min_cluster_size` (reusing the cached embeddings) to show that sensitivity. Off by default —
set `RUN_SWEEP = True` to run (a few minutes; refits UMAP+HDBSCAN per setting).

In [14]:
RUN_SWEEP = False
if RUN_SWEEP:
    sweep = []
    for mcs in [15, 30, 60, 120]:
        tm = BERTopic(
            umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                            metric="cosine", random_state=RANDOM_STATE),
            hdbscan_model=HDBSCAN(min_cluster_size=mcs, metric="euclidean",
                                  cluster_selection_method="eom"),
            vectorizer_model=CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=5),
            ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
            calculate_probabilities=False, verbose=False,
        )
        tt = np.asarray(tm.fit_transform(docs, embeddings=embeddings)[0])
        sweep.append({"min_cluster_size": mcs,
                      "n_topics": len(set(tt) - {-1}),
                      "outlier_share": f"{np.mean(tt == -1):.1%}"})
    display(pd.DataFrame(sweep))
else:
    print("RUN_SWEEP = False — set to True to run the sweep.")

RUN_SWEEP = False — set to True to run the sweep.
